# ORCA - QLoRA train from ZIP 400 samples (Colab T4)

**How to use**
1. Runtime -> Change runtime type -> **T4 GPU**
2. Run Upload cell -> select zip (must contain train.jsonl + val.jsonl)
3. Runtime -> **Run all**
4. Wait for train -> download orca-analyst-lora.zip

**OOM-safe defaults (T4 16GB):** max_seq_length=1024, gradient_checkpointing=True, LoRA r=16, eval off during train.


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Enable Runtime -> T4 GPU first'
print('GPU:', torch.cuda.get_device_name(0))
torch.cuda.empty_cache()


In [ ]:
!pip install -q -U transformers==4.51.3 peft==0.15.2 trl==0.15.2 bitsandbytes==0.45.4 accelerate datasets sentencepiece protobuf


## 1. Upload ZIP (400 samples)

Run this cell then pick the zip file containing train.jsonl and val.jsonl.


In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

print('Select ZIP file to upload...')
uploaded = files.upload()
assert uploaded, 'No file uploaded'

zip_name = list(uploaded.keys())[0]
print('Uploaded:', zip_name)

extract_dir = Path('orca_data')
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall(extract_dir)
    print('Files:', zf.namelist())

train_path = None
val_path = None
for p in extract_dir.rglob('*.jsonl'):
    name = p.name.lower()
    if name == 'train.jsonl':
        train_path = p
    elif name == 'val.jsonl':
        val_path = p

assert train_path is not None, 'train.jsonl not found in zip'
print('train:', train_path)
print('val:', val_path if val_path else '(none - train only)')

n_train = sum(1 for line in open(train_path, encoding='utf-8') if line.strip())
n_val = sum(1 for line in open(val_path, encoding='utf-8') if line.strip()) if val_path else 0
print('N train:', n_train, '| N val:', n_val)


## 2. Load HuggingFace dataset


In [ ]:
from datasets import load_dataset

data_files = {'train': str(train_path)}
if val_path is not None:
    data_files['validation'] = str(val_path)

raw = load_dataset('json', data_files=data_files)
print(raw)

sample = raw['train'][0]
print('keys:', sample.keys())
print('roles:', [m['role'] for m in sample['messages']])
print('assistant preview:', sample['messages'][-1]['content'][:200])


## 3. Load Qwen2.5-7B 4-bit + LoRA (r=16, OOM-safe for T4)

If you have A100 40GB you can set r=32 and max_seq_length=2048 in the train cell.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE = 'Qwen/Qwen2.5-7B-Instruct'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
torch.cuda.empty_cache()


## 4. Train (3 epochs, OOM-safe settings for T4 16GB)


In [ ]:
from trl import SFTTrainer, SFTConfig
import torch

def formatting_func(example):
    return tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )

args = SFTConfig(
    output_dir='./orca-400-out',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1.5e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='no',
    bf16=True,
    optim='paged_adamw_8bit',
    max_seq_length=1024,
    packing=False,
    report_to='none',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    dataloader_pin_memory=False,
)

torch.cuda.empty_cache()

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=raw['train'],
    processing_class=tokenizer,
    formatting_func=formatting_func,
)
trainer.train()
print('TRAIN DONE')


## 5. Save adapter + download


In [ ]:
OUT = 'orca-analyst-lora'
model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)

!zip -r orca-analyst-lora.zip orca-analyst-lora
!ls -lh orca-analyst-lora.zip

from google.colab import files
files.download('orca-analyst-lora.zip')
print('Downloaded orca-analyst-lora.zip - next: host with vLLM')


## 6. Deploy vLLM (GPU server, not Colab)

```bash
unzip orca-analyst-lora.zip
vllm serve Qwen/Qwen2.5-7B-Instruct \
  --enable-lora \
  --lora-modules orca-analyst-v1=./orca-analyst-lora \
  --host 0.0.0.0 --port 8000 --max-model-len 4096
```

ORCA `.env`:
```bash
AI_BASE_URL=https://YOUR_HOST/v1
AI_MODEL_ANALYSIS=orca-analyst-v1
AI_MODEL_ANALYSIS_FALLBACKS=qwen/qwen3.8-27b:free,inclusionai/ling-3.0-flash-fin:free
```

If still OOM: set max_seq_length=768 or use a GPU with more VRAM.
